# Legacy 저장 모델 예측

특징 추출 노트북과 `legacy/models`의 저장 모델을 사용합니다. 거리 기반 점수, 최고·최저값 제외 평균, 70% 임계값은 원본과 동일합니다.


In [ ]:
from pathlib import Path


def find_project_root(start):
    """현재 실행 위치에서 프로젝트 루트를 찾습니다."""
    current = Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "legacy" / "notebooks" / "최종.ipynb").exists():
            return candidate
    raise FileNotFoundError("프로젝트 루트를 찾을 수 없습니다.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "legacy" / "전체15.csv"
MODEL_DIR = PROJECT_ROOT / "legacy" / "models"
IMAGE_DIR = PROJECT_ROOT / "legacy" / "images"


In [ ]:
feature_notebook = PROJECT_ROOT / "legacy" / "notebooks" / "01_legacy_feature_extraction.ipynb"
get_ipython().run_line_magic("run", str(feature_notebook))


## 저장 모델 로드


In [ ]:
# 모델 로드 및 매핑 설정
kmeans = joblib.load(MODEL_DIR / 'kmeans.pkl')
gmm = joblib.load(MODEL_DIR / 'gmm.pkl')
meanshift = joblib.load(MODEL_DIR / 'meanshift.pkl')
agglomerative = joblib.load(MODEL_DIR / 'agglomerative.pkl')
log_reg = joblib.load(MODEL_DIR / 'lr.pkl')
rf = joblib.load(MODEL_DIR / 'rf.pkl')
# dbscan = joblib.load(MODEL_DIR / 'dbscan.pkl')

# 변수 로드
cluster_labels = joblib.load(MODEL_DIR / 'cluster_labels.pkl')

# 원본 predict_phishing 함수가 사용하는 학습 데이터
data = pd.read_csv(DATA_PATH)
X = data.drop(['url', 'label'], axis=1)
y = data['label']


## 예측 함수


In [ ]:
def calculate_phishing_probability(features_array, cluster_centers):
    # 각 군집 대표점과의 거리 계산
    distances = [euclidean(features_array, rep) for rep in cluster_centers]

    # 거리를 확률로 변환 (Softmax 사용)
    probabilities = np.exp(-np.array(distances)) / np.sum(np.exp(-np.array(distances)))

    # 군집 레이블과 확률 연결
    cluster_probabilities = {label: prob for label, prob in enumerate(probabilities)}

    # 정상 및 피싱 확률 계산 (군집 0을 정상, 군집 1을 피싱으로 가정)
    normal_prob = cluster_probabilities.get(0, 0)
    phishing_prob = cluster_probabilities.get(1, 0)

    return phishing_prob * 100



def is_valid_url(url):
    pattern = re.compile(r'^(https?://|www\.|WWW\.)[^\s/$.?#].[^\s]*$', re.IGNORECASE)
    return bool(pattern.match(url))



def calculate_cluster_representatives(X, labels):
    n_clusters = len(np.unique(labels))
    representatives = []
    for i in range(n_clusters):
        cluster_points = X[labels == i]
        representative = np.mean(cluster_points, axis=0)
        representatives.append(representative)
    return np.array(representatives)



def predict_phishing(url):
    try:
        if not is_valid_url(url):
            return "오류 발생", {}

        # URL을 특성 벡터로 변환
        features_df, features_array = feature_extract(url)
        features_array = features_array.reshape(1, -1)  # Reshape to 2D array

        # 각 모델에 대한 클러스터 예측
        kmeans_cluster = kmeans.predict(features_array)[0]
        gmm_cluster = gmm.predict(features_array)[0]
        meanshift_cluster = meanshift.predict(features_array)[0]
        # dbscan_cluster = dbscan.fit_predict(features_array)[0]

        # 군집별 중심값
        kmeans_center = kmeans.cluster_centers_
        gmm_center = gmm.means_
        meanshift_center = meanshift.cluster_centers_
        agglomerative_center = calculate_cluster_representatives(X, cluster_labels)

        # Softmax를 사용한 클러스터 확률 계산
        kmeans_prob = calculate_phishing_probability(features_array, kmeans_center)
        gmm_prob = calculate_phishing_probability(features_array, gmm_center)
        meanshift_prob = calculate_phishing_probability(features_array, meanshift_center)
        agglomerative_prob = calculate_phishing_probability(features_array, agglomerative_center)

        # 로지스틱 회귀와 랜덤 포레스트 예측
        log_reg_prob = log_reg.predict_proba(features_df)[0][1] * 100
        rf_prob = rf.predict_proba(features_df)[0][1] * 100

        # 확률 값들을 리스트에 저장
        cluster_probs = [kmeans_prob, gmm_prob, meanshift_prob, agglomerative_prob]

        # 최고값과 최저값을 제거한 나머지 값들
        cluster_probs.remove(max(cluster_probs))  # 최고값 제거
        cluster_probs.remove(min(cluster_probs))  # 최저값 제거

        # 남은 2개의 값의 합을 구한 후 2로 나누어 평균 계산
        average_cluster_prob = (cluster_probs[0] + cluster_probs[1]) / 2

        # 피싱 여부 결정
        site_status = "피싱 사이트" if average_cluster_prob > 70 else "정상 사이트"

        # 예측 결과 출력
        result = f"{site_status} 입니다."

        return result, features_df

    except Exception as e:
        print(f"Error during prediction: {str(e)}")
        return "오류 발생", {}
